# SVM & Vectorization Demo

This notebook walks through:
1. **What is Vectorization?** — Turning text into numbers a machine can understand
2. **How the same word gets different vector values** depending on context
3. **How SVM (Support Vector Machine) uses those vectors** to classify text
4. **Image Classification with SVM** — A visual example using pixel data

---
## Step 0: Install & Import Libraries
Run this cell first to make sure everything is available.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

print("All imports successful!")

---
## Part 1: Two Sentences, One Common Word

Let's take two simple sentences that share the word **"valve"** but mean very different things:

| Sentence | Intended Category |
|----------|------------------|
| `"valve assembly for compressor unit"` | Compressor Parts |
| `"valve stem seal kit for refrigerant line"` | Refrigerant Parts |

Both contain **"valve"**, but the surrounding words change how the machine sees them.

In [ ]:
# Our two sentences with the common word "valve"
sentence_1 = "valve assembly for compressor unit"
sentence_2 = "valve stem seal kit for refrigerant line"

sentences = [sentence_1, sentence_2]

print("Sentence 1:", sentence_1)
print("Sentence 2:", sentence_2)
print("\nCommon word: 'valve'")

---
## Part 2: CountVectorizer — Simple Word Counting

The simplest approach: count how many times each word appears.

Each sentence becomes a row of numbers — one number per unique word in the entire vocabulary.

In [ ]:
# Step 1: Fit a CountVectorizer on both sentences
count_vec = CountVectorizer()
count_matrix = count_vec.fit_transform(sentences)

# Show the vocabulary (each word gets an index)
vocab = count_vec.get_feature_names_out()
print("Vocabulary (all unique words):")
print(vocab)
print(f"\nTotal unique words: {len(vocab)}")

In [ ]:
# Step 2: Show the vector output for each sentence
count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=vocab,
    index=["Sentence 1", "Sentence 2"]
)

print("=" * 60)
print("COUNT VECTORIZER OUTPUT")
print("=" * 60)
print(count_df.to_string())
print("\n--- Key Observation ---")
print(f"'valve' column: Sentence 1 = {count_df.loc['Sentence 1', 'valve']}, Sentence 2 = {count_df.loc['Sentence 2', 'valve']}")
print("Both sentences have the SAME value for 'valve' (1).")
print("The difference comes from the OTHER words.")

---
## Part 3: TF-IDF Vectorizer — Weighted Word Importance

**TF-IDF** = Term Frequency × Inverse Document Frequency

- **TF**: How often a word appears in THIS sentence
- **IDF**: How rare the word is across ALL sentences

Words that appear in EVERY sentence (like "valve" and "for") get **lower scores**.  
Words unique to one sentence get **higher scores**.

This is exactly what our SVM production code uses (`TfidfVectorizer` + `SVC`).

In [ ]:
# Step 1: Fit a TF-IDF Vectorizer on both sentences
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(sentences)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(4),
    columns=tfidf_vec.get_feature_names_out(),
    index=["Sentence 1", "Sentence 2"]
)

print("=" * 60)
print("TF-IDF VECTORIZER OUTPUT")
print("=" * 60)
print(tfidf_df.to_string())
print("\n--- Key Observation ---")
print(f"'valve' in Sentence 1: {tfidf_df.loc['Sentence 1', 'valve']:.4f}")
print(f"'valve' in Sentence 2: {tfidf_df.loc['Sentence 2', 'valve']:.4f}")
print(f"'for'   in Sentence 1: {tfidf_df.loc['Sentence 1', 'for']:.4f}")
print(f"'for'   in Sentence 2: {tfidf_df.loc['Sentence 2', 'for']:.4f}")
print("\nShared words ('valve', 'for') get LOWER TF-IDF scores because they appear in both sentences.")
print("Unique words ('compressor', 'refrigerant', etc.) get HIGHER scores — they are more informative.")

---
## Part 4: Side-by-Side Comparison

Let's visually compare how the two vectorizers treat the **same sentences**.

In [ ]:
# Side-by-side comparison for Sentence 1
compare_s1 = pd.DataFrame({
    "Word": vocab,
    "CountVec (Sentence 1)": count_matrix.toarray()[0],
    "TF-IDF (Sentence 1)": tfidf_matrix.toarray()[0].round(4)
})

compare_s2 = pd.DataFrame({
    "Word": vocab,
    "CountVec (Sentence 2)": count_matrix.toarray()[1],
    "TF-IDF (Sentence 2)": tfidf_matrix.toarray()[1].round(4)
})

print("=" * 50)
print("SENTENCE 1: 'valve assembly for compressor unit'")
print("=" * 50)
print(compare_s1.to_string(index=False))

print("\n" + "=" * 50)
print("SENTENCE 2: 'valve stem seal kit for refrigerant line'")
print("=" * 50)
print(compare_s2.to_string(index=False))

print("\n--- Summary ---")
print("CountVec: All present words = 1, absent words = 0 (no weighting)")
print("TF-IDF:   Shared words get LOWER scores, unique words get HIGHER scores")
print("\nThis is WHY TF-IDF helps SVM distinguish between similar descriptions!")

In [ ]:
# Visual bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CountVectorizer
x = np.arange(len(vocab))
width = 0.35
axes[0].bar(x - width/2, count_matrix.toarray()[0], width, label='Sentence 1', color='steelblue')
axes[0].bar(x + width/2, count_matrix.toarray()[1], width, label='Sentence 2', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(vocab, rotation=45, ha='right')
axes[0].set_title('CountVectorizer Output')
axes[0].set_ylabel('Count')
axes[0].legend()

# TF-IDF
axes[1].bar(x - width/2, tfidf_matrix.toarray()[0], width, label='Sentence 1', color='steelblue')
axes[1].bar(x + width/2, tfidf_matrix.toarray()[1], width, label='Sentence 2', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(vocab, rotation=45, ha='right')
axes[1].set_title('TF-IDF Vectorizer Output')
axes[1].set_ylabel('TF-IDF Score')
axes[1].legend()

plt.suptitle('Same Sentences, Different Vectorization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: In TF-IDF, shared words ('valve', 'for') have EQUAL but REDUCED scores.")
print("The unique words stand out more — this helps SVM draw a better decision boundary.")

---
## Part 5: Watch the Vector Change When We Add More Documents

TF-IDF scores are **relative to the entire corpus**. Adding more sentences changes the vectors.

Let's see what happens when we add a third sentence.

In [ ]:
# Original: 2 sentences
corpus_small = [
    "valve assembly for compressor unit",
    "valve stem seal kit for refrigerant line"
]

# Expanded: add a third sentence that also mentions "valve"
corpus_expanded = [
    "valve assembly for compressor unit",
    "valve stem seal kit for refrigerant line",
    "valve pressure regulator for cooling system"
]

# Vectorize both
tfidf_small = TfidfVectorizer()
tfidf_expanded = TfidfVectorizer()

matrix_small = tfidf_small.fit_transform(corpus_small)
matrix_expanded = tfidf_expanded.fit_transform(corpus_expanded)

# Show "valve" score for Sentence 1 in both cases
valve_idx_small = list(tfidf_small.get_feature_names_out()).index("valve")
valve_idx_expanded = list(tfidf_expanded.get_feature_names_out()).index("valve")

print("=" * 60)
print("HOW 'valve' SCORE CHANGES FOR SENTENCE 1")
print("=" * 60)
print(f"With 2 sentences: valve = {matrix_small.toarray()[0][valve_idx_small]:.4f}")
print(f"With 3 sentences: valve = {matrix_expanded.toarray()[0][valve_idx_expanded]:.4f}")
print("\nThe score DECREASED because 'valve' now appears in ALL 3 sentences.")
print("TF-IDF automatically reduces the weight of common words.")
print("\nThis is why our production model retrains when new reference data is added —")
print("the vectors shift as the corpus grows.")

---
## Part 6: SVM in Action — Text Classification

Now let's see the full pipeline: **TF-IDF + SVM** classifying product descriptions.

This mirrors what our production code does in Block-2.

In [ ]:
# Sample training data (product descriptions and their categories)
train_descriptions = [
    "compressor assembly valve kit",
    "compressor motor bearing replacement",
    "compressor discharge valve plate",
    "compressor scroll set high pressure",
    "refrigerant valve service port adapter",
    "refrigerant recovery unit hose set",
    "refrigerant charging scale digital",
    "refrigerant leak detector sensor probe",
    "filter drier core replacement cartridge",
    "filter housing gasket seal kit",
    "filter suction line strainer mesh",
    "filter air intake panel washable"
]

train_labels = [
    "Compressor", "Compressor", "Compressor", "Compressor",
    "Refrigerant", "Refrigerant", "Refrigerant", "Refrigerant",
    "Filtration", "Filtration", "Filtration", "Filtration"
]

print("Training Data:")
for desc, label in zip(train_descriptions, train_labels):
    print(f"  [{label:12s}] {desc}")

In [ ]:
# Build the pipeline (same as production: TfidfVectorizer + SVC)
model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))

# Train the model
model.fit(train_descriptions, train_labels)
print("Model trained successfully!")
print(f"Pipeline steps: {[step[0] for step in model.steps]}")

In [ ]:
# Test with new descriptions the model has NEVER seen
test_descriptions = [
    "valve assembly for compressor unit",          # Has 'valve' + 'compressor'
    "valve stem seal kit for refrigerant line",    # Has 'valve' + 'refrigerant'
    "filter replacement cartridge drier",           # filter + drier
    "compressor oil separator element",             # compressor context
    "refrigerant pressure gauge manifold set"       # refrigerant context
]

print("=" * 70)
print("PREDICTIONS ON NEW DESCRIPTIONS")
print("=" * 70)

for desc in test_descriptions:
    prediction = model.predict([desc])[0]
    probabilities = model.predict_proba([desc])[0]
    confidence = max(probabilities) * 100
    
    print(f"\nInput:       '{desc}'")
    print(f"Predicted:   {prediction}")
    print(f"Confidence:  {confidence:.1f}%")
    print(f"All scores:  {dict(zip(model.classes_, [f'{p:.1%}' for p in probabilities]))}")

print("\n" + "=" * 70)
print("Notice: Both sentences contain 'valve', but SVM correctly classifies")
print("them into DIFFERENT categories based on the surrounding context words.")
print("This is the power of TF-IDF + SVM working together.")

---
## Part 6b: Visualizing Support Vectors per Category

SVM works by finding **support vectors** — the training samples closest to the decision boundary.  
These are the critical data points that define where one category ends and another begins.

Since our TF-IDF vectors are high-dimensional, we use **PCA** to project them down to 2D so we can see:
- Where each training document sits in vector space
- Which ones are the **support vectors** (circled)
- Where the **decision boundaries** fall between categories

In [ ]:
from sklearn.decomposition import PCA

# Extract the trained components from the pipeline
vectorizer = model.named_steps['tfidfvectorizer']
svm_model = model.named_steps['svc']

# Transform training data to TF-IDF vectors
X_train_tfidf = vectorizer.transform(train_descriptions)

# Reduce to 2D with PCA for visualization
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_train_tfidf.toarray())

# Get support vector indices
support_indices = svm_model.support_

# Color map for categories
categories = list(set(train_labels))
color_map = {'Compressor': 'steelblue', 'Refrigerant': 'coral', 'Filtration': 'seagreen'}
marker_map = {'Compressor': 'o', 'Refrigerant': 's', 'Filtration': '^'}

fig, ax = plt.subplots(figsize=(10, 7))

# Plot decision regions (background shading)
h = 0.05
x_min, x_max = X_2d[:, 0].min() - 0.3, X_2d[:, 0].max() + 0.3
y_min, y_max = X_2d[:, 1].min() - 0.3, X_2d[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# To shade regions, we need to inverse-map from 2D back — instead we train a quick
# SVM on the 2D PCA projection to show the boundary visually
from sklearn.svm import SVC as SVC2
svm_2d = SVC2(kernel='linear', probability=True)
label_to_num = {label: i for i, label in enumerate(categories)}
y_numeric = np.array([label_to_num[l] for l in train_labels])
svm_2d.fit(X_2d, y_numeric)

Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Light background shading for decision regions
bg_colors = ['#d0e1f9', '#f9d0d0', '#d0f0d0']  # blue, red, green tints
from matplotlib.colors import ListedColormap
ax.contourf(xx, yy, Z, alpha=0.15, colors=bg_colors[:len(categories)])
ax.contour(xx, yy, Z, colors='gray', linewidths=0.5, alpha=0.5)

# Plot each category's training points
for cat in categories:
    mask = np.array([l == cat for l in train_labels])
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=color_map[cat], marker=marker_map[cat], s=100,
               label=f'{cat} (training)', edgecolors='black', linewidth=0.5, zorder=3)

# Highlight support vectors with large circles
ax.scatter(X_2d[support_indices, 0], X_2d[support_indices, 1],
           s=300, facecolors='none', edgecolors='red', linewidths=2,
           label='Support Vectors', zorder=4)

# Label each point with its description (abbreviated)
for i, desc in enumerate(train_descriptions):
    short = desc[:20] + "..." if len(desc) > 20 else desc
    is_sv = "(SV) " if i in support_indices else ""
    ax.annotate(f'{is_sv}{short}', (X_2d[i, 0], X_2d[i, 1]),
                fontsize=7, ha='left', va='bottom',
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_title('SVM Support Vectors by Category\n(Red circles = Support Vectors that define the decision boundary)', fontsize=12)
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nTotal training samples: {len(train_descriptions)}")
print(f"Support vectors selected: {len(support_indices)} out of {len(train_descriptions)}")
print(f"\nSupport vector descriptions (the critical boundary-defining samples):")
for idx in support_indices:
    print(f"  [{train_labels[idx]:12s}] {train_descriptions[idx]}")

---
## Part 6c: How a New Document Gets Classified (Live Example)

When a **new, unseen description** arrives, here's what happens step by step:
1. **Vectorize** — TF-IDF converts the text to a numeric vector using the learned vocabulary
2. **Project** — The vector lands at a specific point in the feature space
3. **Classify** — SVM checks which side of the decision boundary it falls on
4. **Confidence** — Distance from the boundary determines confidence

The chart below shows the new documents (stars) plotted alongside the training data, so you can see **exactly why** each one gets its predicted category.

In [ ]:
# Vectorize the new test descriptions using the SAME fitted vectorizer
X_test_tfidf = vectorizer.transform(test_descriptions)
X_test_2d = pca.transform(X_test_tfidf.toarray())

# Get predictions for test descriptions
test_predictions = model.predict(test_descriptions)
test_probas = model.predict_proba(test_descriptions)

fig, ax = plt.subplots(figsize=(12, 8))

# Decision boundary background (reuse from Part 6b)
ax.contourf(xx, yy, Z, alpha=0.12, colors=bg_colors[:len(categories)])
ax.contour(xx, yy, Z, colors='gray', linewidths=0.5, alpha=0.4)

# Plot training points (smaller, faded)
for cat in categories:
    mask = np.array([l == cat for l in train_labels])
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=color_map[cat], marker=marker_map[cat], s=60, alpha=0.4,
               edgecolors='black', linewidth=0.3, zorder=2)

# Highlight support vectors
ax.scatter(X_2d[support_indices, 0], X_2d[support_indices, 1],
           s=200, facecolors='none', edgecolors='red', linewidths=1.5,
           label='Support Vectors', zorder=3, alpha=0.6)

# Plot NEW test documents as large stars
for i, (desc, pred) in enumerate(zip(test_descriptions, test_predictions)):
    conf = max(test_probas[i]) * 100
    ax.scatter(X_test_2d[i, 0], X_test_2d[i, 1],
               c=color_map[pred], marker='*', s=400,
               edgecolors='black', linewidth=1.5, zorder=5)
    short = desc[:25] + "..." if len(desc) > 25 else desc
    ax.annotate(f'NEW: "{short}"\n-> {pred} ({conf:.0f}%)',
                (X_test_2d[i, 0], X_test_2d[i, 1]),
                fontsize=8, fontweight='bold', ha='left', va='bottom',
                xytext=(8, 8), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# Draw lines from new docs to their nearest support vector
for i in range(len(test_descriptions)):
    # Find closest support vector
    dists = np.sqrt(((X_2d[support_indices] - X_test_2d[i]) ** 2).sum(axis=1))
    nearest_sv_idx = support_indices[np.argmin(dists)]
    ax.plot([X_test_2d[i, 0], X_2d[nearest_sv_idx, 0]],
            [X_test_2d[i, 1], X_2d[nearest_sv_idx, 1]],
            'k--', alpha=0.3, linewidth=1)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker=marker_map[cat], color='w', markerfacecolor=color_map[cat],
           markersize=8, label=f'{cat} (training)') for cat in categories
] + [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='gold',
           markersize=15, markeredgecolor='black', label='New Document (live)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
           markersize=12, markeredgecolor='red', markeredgewidth=2, label='Support Vector'),
]
ax.legend(handles=legend_elements, loc='best', fontsize=9)

ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_title('How New Documents Get Classified\n(Stars = new inputs, landing in the region defined by support vectors)',
             fontsize=12)
plt.tight_layout()
plt.show()

print("=" * 70)
print("HOW CLASSIFICATION WORKS:")
print("=" * 70)
print("1. The new document is vectorized using TF-IDF (same vocabulary as training)")
print("2. It lands at a point in the vector space (shown as a star)")
print("3. SVM checks which side of the decision boundary that point falls on")
print("4. The support vectors (red circles) are the training points that DEFINE")
print("   these boundaries — they are the closest points between categories")
print("5. Confidence = how far the new point is from the boundary")
print("   (further from boundary = higher confidence)")
print("\nDashed lines connect each new document to its nearest support vector,")
print("showing which training example most influenced the classification.")

---
## Summary

| Concept | What It Does | Example |
|---------|-------------|--------|
| **CountVectorizer** | Counts word occurrences | `"valve" → 1` (same in both sentences) |
| **TF-IDF Vectorizer** | Weights words by rarity | `"valve" → 0.33` (lower because shared) |
| **SVM** | Finds the best dividing line between categories | Separates Compressor vs Refrigerant parts |
| **Support Vectors** | Training points closest to the boundary | The few samples that define where one category ends and another begins |

### Key Takeaways
1. **Same word, different vectors** — TF-IDF gives context-aware weights; shared words score lower
2. **Vectors change when corpus changes** — Adding more documents shifts all TF-IDF scores
3. **SVM finds the optimal separating hyperplane** — maximizing the margin between categories
4. **Support vectors are the key** — Only a subset of training samples actually define the boundary; the rest could be removed without changing the model
5. **New documents get classified by position** — A new input is vectorized and lands on one side of the boundary; its distance from the boundary determines confidence
6. **Our production pipeline** uses `TfidfVectorizer() + SVC(kernel='linear')` to classify part descriptions into taxonomy nodes